<a href="https://colab.research.google.com/github/de-Zest/python-ai-governance/blob/main/phase2_data_handling_02_full_eval_pipeline.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
# Phase 2 - Full Eval Pipeline
# Goal: Complete LLM evaluation pipeline with word count, timing and keyword scoring
# Date: May 2026
# Status: In progress
import time
import pandas as pd
import os
from google import genai
from google.colab import userdata, drive

# Mount Drive
drive.mount('/content/drive')
SAVE_PATH = "/content/drive/MyDrive/python-ai-governance/data/"
os.makedirs(SAVE_PATH, exist_ok=True)

# Setup
client = genai.Client(api_key=userdata.get('GOOGLE_API_KEY'))

# Function 1 - Ask LLM with timing
def ask_llm(prompt):
  start = time.time()
  response = client.models.generate_content(
      model="gemini-flash-latest",
      contents=prompt
  )
  end = time.time()
  duration = round(end - start, 2)
  return response.text, duration

#Function 2 - Full eval pipeline
def run_eval(prompt_list, topic="General"):
  print(f"====== EVALUATION: {topic} ======")

  results = []
  governance_keywords = ["risk", "safety", "human", "model", "data", "governance"]

  for i, prompt in enumerate(prompt_list):
    response, duration = ask_llm(prompt)
    word_count = len(response.split())

    # Keyword scoring
    keyword_hits = sum(1 for kw in governance_keywords if kw in response.lower())

    results.append({
        "prompt_number": i + 1,
        "prompt": prompt,
        "response": response,
        "word_count": word_count,
        "response_time": duration,
        "keyword_score": keyword_hits,
        "max_score": len(governance_keywords),
    })

    print(f"[{i+1}] Done - {word_count} words /"
          f"{duration}s / "
          f"keywords {keyword_hits}/{len(governance_keywords)}")
    time.sleep(2)

  df = pd.DataFrame(results)

  # Summary
  print(f"\n====== EVAL SUNNMARY: {topic} ======")
  print(f"Total prompts: {len(df)}")
  print(f"Avg word count: {df['word_count'].mean():.0f}words")
  print(f"Avg response time: {df['response_time'].mean():.2f}s")
  print(f"Avg keyword score: {df['keyword_score'].mean():.1f}/{len(governance_keywords)}")
  print(f"Fastest response: {df['response_time'].min()}s")
  print(f"Slowest response: {df['response_time'].max()}s")

  # Save to Drive
  filepath = SAVE_PATH + "ai_governance_eval_results.csv"
  df.to_csv(filepath, index=False)
  print(f"\nSaved to Google Drive ✅")

  return df

# Prompts
ai_governance_prompts = [
    "What is the key perspective of EU AI ACT?",
    "What is the perspective of NIST AI framework?",
    "What is the need for RLHF in AI model evaluation?",
    "Why does AI governance matter for the future of humanity?",
]

# Run
df = run_eval(ai_governance_prompts, topic="AI Governance Basics")

# Display table
df[["prompt_number", "prompt", "word_count", "response_time", "keyword_score"]]

Mounted at /content/drive
====== EVALUATION: AI Governance Basics ======
[1] Done - 574 words /8.28s / keywords 5/6
[2] Done - 558 words /8.71s / keywords 3/6
[3] Done - 674 words /13.94s / keywords 4/6
[4] Done - 630 words /11.13s / keywords 6/6

====== EVAL SUNNMARY: AI Governance Basics ======
Total prompts: 4
Avg word count: 609words
Avg response time: 10.52s
Avg keyword score: 4.5/6
Fastest response: 8.28s
Slowest response: 13.94s

Saved to Google Drive ✅


,prompt_number,prompt,word_count,response_time,keyword_score
0,1,What is the key perspective of EU AI ACT?,574,8.28,5
1,2,What is the perspective of NIST AI framework?,558,8.71,3
2,3,What is the need for RLHF in AI model evaluation?,674,13.94,4
3,4,Why does AI governance matter for the future o...,630,11.13,6
